Figures for the project scientific article
link to the doi: 

In [ ]:
# Import libraries
import os
import s3fs

# Computations
import numpy as np
import scipy.stats
import pandas as pd
import xarray as xr
import xclim as xc
from xclim.core.indicator import registry
import xscen as xs

# Graphics
import matplotlib
from matplotlib import ticker
import matplotlib.pyplot as plt
import figanos.matplotlib as fg

In [ ]:
# Set options
fg.utils.set_mpl_style('ouranos')
matplotlib.rcParams["legend.title_fontsize"] = 10
matplotlib.rcParams["legend.fontsize"] = 10
matplotlib.rcParams["axes.titlesize"] = 10
matplotlib.rcParams["axes.labelsize"] = 10



xc.set_options(metadata_locales=["fr"])
xr.set_options(keep_attrs=True)

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']


In [ ]:
# Function to read from MinIo the netcdf created for the project (https://www.frdr-dfdr.ca/repo/dataset/876e9380-63fc-4eaa-987b-aa16c3770941)
def read_from_minio(filename: str):
    """Read Zarr object from MinIO and return as xarray dataset.

    Parameters
    ----------
    filename: str
      Path to the Zarr object in the portail-ing bucket on the Ouranos MinIO server.
    """
    s3r = s3fs.S3FileSystem(anon=True, use_ssl=False, client_kwargs={"endpoint_url": "http://minio.ouranos.ca"})
    root = f"portail-ing/{filename}"
    store = s3fs.S3Map(root=root, s3=s3r, check=False)
    return xr.open_zarr(store=store)

In [ ]:
# load datasets inside dictionary
data = {
    "espo": {
        "tas": read_from_minio('portail_ing_tas_CMIP6_stations_AHCCD_concat.zarr')['tas'],
        "tasmin": read_from_minio('portail_ing_tasmin_CMIP6_stations_AHCCD_concat.zarr')['tasmin'],
        "tasmax": read_from_minio('portail_ing_tasmax_CMIP6_stations_AHCCD_concat.zarr')['tasmax'],
        "pr": read_from_minio('portail_ing_pr_CMIP6_stations_AHCCD_concat.zarr')['pr'],
    },
    "ahccd": {
        "tas": xr.open_dataset('https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/station_obs/ECCC_AHCCD_gen3_temperature.ncml')['tas'],
        "tasmin": xr.open_dataset('https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/station_obs/ECCC_AHCCD_gen3_temperature.ncml')['tasmin'],
        "tasmax": xr.open_dataset('https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/station_obs/ECCC_AHCCD_gen3_temperature.ncml')['tasmax'],
        "pr": xr.open_dataset('https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/station_obs/ECCC_AHCCD_gen2_precipitation.ncml')['pr'],
    },
}

In [ ]:
# selected indicators from Xclim registry
indics_reg = [
   # registry["TG_MEAN"](units="degC", parameters=dict(freq="YS"),),
   # registry["HEATING_DEGREE_DAYS"](parameters=dict(thresh="18.0 degC", freq="YS"),),
    registry["COOLING_DEGREE_DAYS"](parameters=dict(thresh="18.0 degC", freq="YS"),),
   # registry['PRCPTOT'](parameters=dict(freq="YS")),
    registry['LIQUIDPRCPTOT'](parameters=dict(freq="YS")),
]

In [ ]:
# required variables for indicators
variables = ["tas", "pr"]

# Find station ID from name and extract data at station for desired variables 
data_sta = {}   
for var in variables:
    names = xr.DataArray(data=data['espo'][var].station.data, coords={"name": data['espo'][var].station_name.data}, dims="name")

    if var=='pr':
        name = 'MONTREAL/PIERRE ELLIOTT TRUDEAU '
    else:
        name = "MONTREAL__TRUDEAU_IN"

    station = names.sel(name=name).item()
    
    for typ in data.keys():
        if typ not in list(data_sta.keys()):
            data_sta[typ] = data[typ][var].sel(station=station).to_dataset()
        else: 
            data_sta[typ][var] = data[typ][var].sel(station=station)

# calculate indicators 
inds = {}
for k, v in data_sta.items():
    indic =  xs.indicators.compute_indicators(v, indics_reg)
    inds[k] = indic['YS-JAN'].load()

In [ ]:
# Import analysis classes and functions
from portail_ing.frontend.parameters import IndicatorObsDA, IndicatorRefDA, IndicatorSimDA, HazardMatrix, HazardThreshold
from portail_ing.risk.priors import scenario_weights_from_iams
from portail_ing.risk.priors import load_sherwood_ecs
from matplotlib.ticker import ScalarFormatter

# Reference and future periods
ref_period = (1981, 2010)
fut_period = (2051, 2080)

# Distribution families
scipy_dists = ["norm", "t", "gamma", "genextreme", "lognorm", "uniform"]

# Metric to pick the best fitting distribution against observations ('bic' or 'aic')
metric = 'bic'

# Significance level for Kolmogorov-Smirnov test
level = 0.05

In [ ]:
# Initiate the IndicatorObsDA class
obs = {}
bics = {}
for k, v in inds['ahccd'].items():
    obs[k] = IndicatorObsDA(data=v, period=ref_period)

    # Create a table with the metric values
    bic = pd.Series(obs[k].metrics_da, index=obs[k].metrics_da.scipy_dist, name=metric.upper())
    bic.sort_values(ascending=True, inplace=True)
    bics[k] = bic

In [ ]:
# Present top three distributions for each indicator based on the metric
fig, axs = plt.subplots(2, 1, figsize=(5,6.5))

n=0
for k  in obs.keys():
    ax = axs[n]

    bic = pd.Series(obs[k].metrics_da, index=obs[k].metrics_da.scipy_dist, name=metric.upper())
    bic.sort_values(ascending=True, inplace=True)
    
    # Histogram
    ax.hist(obs[k].sample, density=True, alpha=.5, rwidth=.9, color='grey', label='Observed')
    x = np.linspace(obs[k].sample.min().item(), obs[k].sample.max().item(), 100)
    x = xr.DataArray(data=x, dims=(obs[k].sample.name,), coords={obs[k].sample.name: x})

    for d in bic.iloc[0:3].index:
        # PDF
        pdf = IndicatorObsDA(data=inds['ahccd'][k], period=ref_period, dist=d).pdf(x)
        ax.plot(x, pdf, label=d)
    ax.set_title(k, loc='left')
    ax.legend(loc='upper right', fontsize='small')
    n += 1

In [ ]:
# Create IndicatorObsDA with new dist, IndicatorRefDA and IndicatorSimDA objects
refs = {}
futs = {}
chosen_dists = {'cooling_degree_days': 'norm', 'liquidprcptot': 'lognorm'}
for k, v in inds['espo'].items():
    obs[k] = IndicatorObsDA(data=inds['ahccd'][k], period=ref_period, dist=chosen_dists[k])
    refs[k] = IndicatorRefDA(data=v, obs=obs[k], level=level, dist=chosen_dists[k])
    futs[k] = IndicatorSimDA(data=v,  period=fut_period, obs=obs[k], model_weights=refs[k].model_weights, dist=chosen_dists[k])

# Cooling-degree-days figure

In [ ]:
cind = 'cooling_degree_days'

### Weigths figure

In [ ]:
from matplotlib import patches as mpatches
import matplotlib as mpl

fig, axs = plt.subplot_mosaic(
    [['(a)'], ['(b)'], ['(c)']], 
    figsize=(5, 6),
    #(5.9, 7.9) good but too big 
    layout='constrained',
)


# Weights (0/1) based on the K-S test
w1 = refs[cind]._ks.to_series() * 1

# Weights based on the ECS and GHG scenario
w2 = refs[cind].model_weights.to_series()

ecs = refs[cind].model_weights.coords['ecs'].values

w = pd.DataFrame({'KS test ($w_1$)': w1, 'ECS ($w_2$)': w2}).copy()
w.index.rename("Models", inplace=True)

ecs_df = load_sherwood_ecs()
pdf = xr.DataArray(ecs_df["pdf"], dims="ecs", coords={"ecs": ecs_df["ECS"]})

# --- ECS figure --- 
pdf.plot(ax=axs['(a)'], color='darkslategray', alpha=0.8, label="ECS PDF", zorder=10, linewidth=2)
axs['(a)'].hist(ecs, density=True, bins=15, color='grey', alpha=0.5, label="Unweighted ECS", rwidth=.9)
axs['(a)'].hist(ecs, density=True, bins=15, color=colors[1], alpha=0.9, label="Weighted ECS",  rwidth=.5, weights=w2)
axs['(a)'].set_xlabel("ECS (°C)")
axs['(a)'].set_ylabel("Probability\ndensity (°C$^{-1}$)")
axs['(a)'].set_xlim(0, 7)
axs['(a)'].legend(loc="upper right", labelspacing=0.25, bbox_to_anchor=(1.23, 1.1), fontsize=10)
#axs['(a)'].set_title("(a) ECS distribution and weights", loc='left')

# --- Model weight figure --- 
ax2 = axs['(b)'].twinx()
b1 = w.plot.bar(y="KS test ($w_1$)", label='w2', color=colors[0], ax=axs['(b)'], zorder=-10, legend=False)
b2 = w.plot.bar(y="ECS ($w_2$)", label='w1', color=colors[1], ax=ax2, legend=False, alpha=.9)
axs['(b)'].set_ylabel("Model performance\nweights ($w_1$)", color=colors[0])
#for label in axs['(b)'].get_xticklabels():
    #label.set_fontsize(8)
axs['(b)'].set_yticks([0, 1])
axs['(b)'].set_ylim(0, 1)
ax2.set_ylabel("Model ECS\nweights ($w_2$)", color=colors[1])
fig.draw_without_rendering()
yt = ax2.get_yticks()
ax2.set_yticks(yt[[0,-1]])
#axs['(b)'].legend(
#    labels=["KS test weights ($w_1)$", "ECS weights ($w_2$)"], 
#    handles=[b1.get_children()[0], 
 #   b2.get_children()[0]], 
  #  loc='lower center', 
   # bbox_to_anchor=(0.8, -1.17), 
    #labelspacing=0.25,
    #ncols=2
  #  )

patch1 = mpatches.Rectangle(
    xy=(-0.09, 0.9),  # x, y in axes fraction coordinates
    width=0.02, height=0.11,
    transform=axs['(b)'].transAxes,
    color=colors[0],
    clip_on=False,
    zorder=10
)
patch2 = mpatches.Rectangle(
    xy=(1.17, 0.9),  # x, y in axes fraction coordinates
    width=0.02, height=0.11,
    transform=axs['(b)'].transAxes,
    color=colors[1],
    clip_on=False,
    zorder=10
)


# Add the patch to the axis
axs['(b)'].add_patch(patch1)
axs['(b)'].add_patch(patch2)

#axs['(b)'].set_title("(b) Model weights based on ECS and K-S test", loc='left')
axs['(b)'].set_xlabel("")

# --- SSP weights ---
import datetime as dt
# Scenarios weights
scen_weights = scenario_weights_from_iams()

# Compute over the future period
w3 = futs[cind].scenario_weights

t = matplotlib.dates.date2num(dt.datetime(fut_period[0], 1, 1)), matplotlib.dates.date2num(dt.datetime(fut_period[1], 1, 1))

ssp_label = {
    "ssp126":"SSP1-2.6",
    "ssp245":"SSP2-4.5",
    "ssp370": "SSP3-7.0",
    "ssp585": "SSP5-8.5"
}

ssp_color = {
    "ssp126": (23, 60, 102),
    "ssp245": (247, 148, 32),
    "ssp370": (231, 29, 37),
    "ssp585": (149, 27,30)
}
axs['(c)'].plot(t, [-1, -1], color='grey', lw=1, ls='--', label="SSP\nweights ($w_3$)")

ssp_color = {k: tuple(v[i] / 255 for i in range(3)) for k, v in ssp_color.items()}
for ssp in scen_weights.experiment_id.values:
    scen_weights.sel(experiment_id=ssp).plot(color=ssp_color[ssp], ax=axs['(c)'], label=None)
    y = w3.sel(experiment_id=ssp).values
    axs['(c)'].plot(t, [y, y], ls="--", color=ssp_color[ssp])

    trans = mpl.transforms.blended_transform_factory(axs['(c)'].transAxes, axs['(c)'].transData)
    axs['(c)'].text(
        1.01,
        scen_weights.sel(experiment_id=ssp).values[-1],
        ssp_label[ssp],
        color=ssp_color[ssp],
        va='center',
        ha='left',
        fontsize=10,
        transform=trans,
        zorder=100,
    )
    
axs['(c)'].axvspan(t[0], t[1], color="#F4E6E6")
axs['(c)'].text(np.mean(t), .6, 'Future period', ha='center', va='center')
axs['(c)'].set_xlabel("")
axs['(c)'].set_ylim(0)
axs['(c)'].grid(False)
axs['(c)'].legend(
    loc='upper left',  
    bbox_to_anchor=(-0.01, 1.05), 
    facecolor='none', 
    frameon=False,
    fontsize=10
)
axs['(c)'].set_title("")
axs['(c)'].set_ylabel("Likelihood of \nSSP scenarios (PgC)")

axs['(c)'].tick_params(axis='both', which='major', labelsize=10)
axs['(b)'].tick_params(axis='both', which='major', labelsize=8)
axs['(a)'].tick_params(axis='both', which='major', labelsize=10)

axs['(a)'].text(0.1, 0.75, "(a)")
axs['(b)'].text(0.1, 1.1, "(b)")
axs['(c)'].text(0.05, 0.93, "(c)", 
    transform=axs['(c)'].transAxes,           # use axes fraction coordinates
    ha='center', va='bottom'          # horizontal/vertical alignment
)


plt.savefig('cdd_18degC_weights.pdf', format='pdf', dpi=600)

In [ ]:
# Figure data
import json

data = {}
for patch in axs['(a)'].patches:
    if patch.get_label() != '_nolegend_':
        data[patch.get_label()] = {
            'x': [patch.get_x().item()],
            'y': [patch.get_height().item()],
        }
        lab = patch.get_label()
        print(patch.get_label())
    # Each patch is a rectangle/bar; you can get its position and height
    data[lab]['x'].append(patch.get_x().item())
    data[lab]['y'].append(patch.get_height().item())

for line in axs['(a)'].get_lines():
    data[line.get_label()] = {
        'x': line.get_xdata().tolist(),
        'y': line.get_ydata().tolist(),
    }

data2 = w.to_dict()

data3 = {}
for line in axs['(c)'].get_lines():
    if line.get_label() in ['SSP1-2.6', 'SSP2-4.5', 'SSP3-7.0', 'SSP5-8.5']:
       data3[line.get_label()] = {
           'x': line.get_xdata().astype(str).tolist(),
           'y': line.get_ydata().tolist(),
       }


# Save extracted data dictionaries to JSON files
with open('ECS.json', 'w') as f:
    json.dump(data, f, indent=2)

with open('cdd_ks_ecs_per_model.json', 'w') as f:
    json.dump(data2, f, indent=2)

with open('SSP_likelihood.json', 'w') as f:
    json.dump(data3, f, indent=2)

### Timeseries and mixtures PDF

In [ ]:
from portail_ing.risk.priors import load_sherwood_ecs
from matplotlib.ticker import ScalarFormatter
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

fig, axs = plt.subplot_mosaic(
    [['(a)'], ['(b)']], 
    figsize=(5.9, 5),
    layout='constrained',
)


percentiles = [10, 50, 90]

#---- plot timeseries ----
plot = {'Observations': inds['ahccd'][cind].sel(time=slice("1950", "2020"))}

exp_per = futs[cind].experiment_percentiles(percentiles)

# Historical period
plot['Historical'] = exp_per.sel(time=slice('1950', '2015'), experiment_id='ssp245')

# Simulations by SSP
groups = inds['espo'][cind].groupby('experiment_id')
for label, da in groups:
        plot[label] = exp_per.sel(time=slice('2015', '2100'), experiment_id=label)
    

# Use Figanos to plot the different time series
fg.plot.timeseries(
    plot,
    ax=axs['(a)'],
    show_lat_lon=False,
    plot_kw= {
        'Observations': {'linestyle': '--', 'color': 'k'},
        'Historical': {'color': '#18BBBB',},
    },
    legend=None,
)

#ax.set_title(title, loc="left")
#axs['(a)'].set_title("a) Timeseries", loc='left')
axs['(a)'].set_ylabel("Cooling degree-days (°C)")
#axs['(a)'].set_xlabel("Time")

handles, labels = axs['(a)'].get_legend_handles_labels()

#add description
handles.append(mlines.Line2D([], [], color='gray'))
labels.append('50th percentile')
handles.append(mpatches.Patch(color='gray', alpha=0.3, linewidth=0))
labels.append('10-90th percentile')
axs['(a)'].legend(handles, labels, loc ='upper left', ncols=2, bbox_to_anchor=(0, 0.95),)


# --- historical histogram + fit dist ----
axs['(b)'].hist(obs[cind].sample, density=True, alpha=.2, rwidth=.9, color=colors[0], 
                 label='Observed histogram',
                 zorder=0,
                )


range_min = refs[cind].sample.min().item() - 0.05 * (refs[cind].sample.max().item() - refs[cind].sample.min().item())
range_max = refs[cind].sample.max().item() + 0.05 * (refs[cind].sample.max().item() - refs[cind].sample.min().item())
x = np.linspace(range_min, range_max, 500)
x = xr.DataArray(data=x, dims=(obs[cind].sample.name,), coords={obs[cind].sample.name: x})

axs['(b)'].plot(x, obs[cind].pdf(x), label='Observed PDF', linewidth=2.5)

#axs['(b)'].set_title('(b) Probability density functions', loc='left')
axs['(b)'].set_ylabel('Probability density (°C$^{-1}$)')



# --- Reference Mixture and pdfs ---

#x = np.linspace(refs[cind].sample.min().item(), refs[cind].sample.max().item(), 100)
vals = refs[cind].pdf(x).values

#Indiviually compute PDF for visualization
from portail_ing.risk.mixture import parametric_pdf
res = parametric_pdf(refs[cind].dparams.stack(realization=('source_id', 'experiment_id')), x)

lab1 = None
lab2 = None

for rea in res.realization.values:
    plt_kw = {
        "alpha": 0.3,
        "linewidth": 0.5,
        "zorder": 1,
    }
    if w.loc[rea[0]]['KS test ($w_1$)'] == 0:
        col =  'darkorange'
        if not lab1:
            lab1 = "Simulated PDF ($w_1$=0) "
            plt_kw['label'] = lab1
    else:
        col = 'grey'
        if not lab2:
            lab2 = "Simulated PDF ($w_1$>0)"
            plt_kw['label'] = lab2    
    plt_kw['color'] = col  

    axs['(b)'].plot(x, res.sel(realization=rea).values, **plt_kw)

axs['(b)'].plot(x, vals, label='Mixture PDF (ref)', linewidth=2.5)


# --- Futre mixture PDF ---

xf = np.linspace(futs[cind].sample.min().item(), futs[cind].sample.max().item(), 100)
valsf = futs[cind].pdf(xf).values
axs['(b)'].plot(xf, valsf, label='Mixture PDF (fut)', linewidth=2.5)
axs['(b)'].axvline(500, color='black', linestyle='--', label=f'Threshold = 500°C', zorder=2)

axs['(b)'].set_xlabel('Cooling degree-days (°C)')
axs['(b)'].set_ylim(bottom=0)
axs['(b)'].set_xlim(75, 1100)

axs['(b)'].legend(loc ='upper right', bbox_to_anchor=(1.05, 1.1))


axs['(b)'].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
axs['(b)'].tick_params(axis='x', pad=5)


axs['(a)'].tick_params(axis='both', which='major', labelsize=10)
axs['(b)'].tick_params(axis='both', which='major', labelsize=10)

axs['(a)'].text(0.05, 0.93, "(a)",    
    transform=axs['(a)'].transAxes,           # use axes fraction coordinates
    ha='center', va='bottom'          # horizontal/vertical alignment
)
axs['(b)'].text(0.05, 0.9, "(b)",    
    transform=axs['(b)'].transAxes,           # use axes fraction coordinates
    ha='center', va='bottom'          # horizontal/vertical alignment
)
#axs['(b)'].text(0.1, 1.1, "(b)")
axs['(a)'].set_title("", loc='left')
axs['(a)'].set_xlabel("")

plt.savefig('cdd_18degC_timeseries.pdf', format='pdf', dpi=600)   

In [ ]:
refs[cind].sample.isel(source_id=0, time=10).dropna(dim='variant_label', how='all')

In [ ]:
refs[cind].sample.isel(source_id=0, time=10).sel(variant_label='r1i1p1f1').values

In [ ]:
refs[cind].sample.variant_label.values

In [ ]:
# figure data 
data = {}
for line in axs['(a)'].get_lines():
    data[line.get_label()] = {
        'x': line.get_xdata().astype(str).tolist(),
        'y': line.get_ydata().tolist(),
    }

data2 = {}
for line in axs['(b)'].get_lines():
    if line.get_label() in ['Observed', 'Mixture (ref)', 'Mixture (fut)']:
        data2[line.get_label()] = {
           'x': line.get_xdata().tolist(),
           'y': line.get_ydata().tolist(),
       }

# Save extracted data dictionaries to JSON files
with open('cdd_timeseries.json', 'w') as f:
    json.dump(data, f, indent=2)

with open('cdd_mixtures_pdf.json', 'w') as f:
    json.dump(data2, f, indent=2)

# Liquid Precipitation (Total rainfall)

In [ ]:
cind = 'liquidprcptot'

## Weights figure

In [ ]:
fig, axs = plt.subplot_mosaic(
    [['(a)'], ['(b)'], ['(c)']], 
    figsize=(5.9, 8),
    layout='constrained',
)


# Weights (0/1) based on the K-S test
w1 = refs[cind]._ks.to_series() * 1

# Weights based on the ECS and GHG scenario
w2 = refs[cind].model_weights.to_series()

ecs = refs[cind].model_weights.coords['ecs'].values

w = pd.DataFrame({'KS test ($w_1$)': w1, 'ECS ($w_2$)': w2}).copy()
w.index.rename("Models", inplace=True)

ecs_df = load_sherwood_ecs()
pdf = xr.DataArray(ecs_df["pdf"], dims="ecs", coords={"ecs": ecs_df["ECS"]})

# --- ECS figure --- 
pdf.plot(ax=axs['(a)'], color='darkslategray', alpha=0.8, label="ECS PDF", zorder=10, linewidth=2)
axs['(a)'].hist(ecs, density=True, bins=15, color='grey', alpha=0.5, label="Unweighted ECS", rwidth=.9)
axs['(a)'].hist(ecs, density=True, bins=15, color=colors[1], alpha=0.9, label="Weighted ECS",  rwidth=.5, weights=w2)
axs['(a)'].set_xlabel("ECS (°C)")
axs['(a)'].set_ylabel("Probability density (°C$^{-1}$)")
axs['(a)'].set_xlim(0, 7)
axs['(a)'].legend(loc="upper left", bbox_to_anchor=(0, 1), labelspacing=0.25,)
axs['(a)'].set_title("(a) ECS distribution and weights", loc='left')

# --- Model weight figure --- 
ax2 = axs['(b)'].twinx()
b1 = w.plot.bar(y="KS test ($w_1$)", label='w2', color=colors[0], ax=axs['(b)'], zorder=-10, legend=False)
b2 = w.plot.bar(y="ECS ($w_2$)", label='w1', color=colors[1], ax=ax2, legend=False, alpha=.9)
axs['(b)'].set_ylabel("Model performance \n weights", color=colors[0])
axs['(b)'].set_yticks([0, 1])
axs['(b)'].set_ylim(0, 1)
ax2.set_ylabel("Model ECS weights", color=colors[1])
fig.draw_without_rendering()
yt = ax2.get_yticks()
ax2.set_yticks(yt[[0,-1]])
axs['(b)'].legend(
    labels=["KS test weights ($w_1)$", "ECS weights ($w_2$)"], 
    handles=[b1.get_children()[0], 
    b2.get_children()[0]], 
    loc='lower center', 
    bbox_to_anchor=(0.8, -1.18), 
    labelspacing=0.25,
    #ncols=2
    )
axs['(b)'].set_title("(b) Model weights based on ECS and K-S test", loc='left')
axs['(b)'].set_xlabel("")

# --- SSP weights ---
import datetime as dt
# Scenarios weights
scen_weights = scenario_weights_from_iams()

# Compute over the future period
w3 = futs[cind].scenario_weights

t = matplotlib.dates.date2num(dt.datetime(fut_period[0], 1, 1)), matplotlib.dates.date2num(dt.datetime(fut_period[1], 1, 1))

ssp_label = {
    "ssp126":"SSP1-2.6",
    "ssp245":"SSP2-4.5",
    "ssp370": "SSP3-7.0",
    "ssp585": "SSP5-8.5"
}

ssp_color = {
    "ssp126": (23, 60, 102),
    "ssp245": (247, 148, 32),
    "ssp370": (231, 29, 37),
    "ssp585": (149, 27,30)
}
axs['(c)'].plot(t, [-1, -1], color='grey', lw=1, ls='--', label="SSP weights ($w_3$)")

ssp_color = {k: tuple(v[i] / 255 for i in range(3)) for k, v in ssp_color.items()}
for ssp in scen_weights.experiment_id.values:
    scen_weights.sel(experiment_id=ssp).plot(color=ssp_color[ssp], ax=axs['(c)'], label=ssp_label[ssp])
    y = w3.sel(experiment_id=ssp).values
    axs['(c)'].plot(t, [y, y], ls="--", color=ssp_color[ssp])
    
axs['(c)'].axvspan(t[0], t[1], color="#F4E6E6")
axs['(c)'].text(np.mean(t), .6, 'Future period', ha='center', va='center')
axs['(c)'].set_xlabel("")
axs['(c)'].set_ylim(0)
axs['(c)'].grid(False)
axs['(c)'].legend(
    loc='upper left',  
    bbox_to_anchor=(0, 1.1), 
    facecolor='none', 
    frameon=False,
    labelspacing=0.25,
    #ncols=2
)
axs['(c)'].set_title("")
axs['(c)'].set_ylabel("Likelihood of \nSSP scenarios (PgC)")

axs['(c)'].set_title("(c) SSP weights", loc='left')
plt.savefig('liquidpr_weights.pdf', format='pdf', dpi=600)

In [ ]:
# Figure data
import json
data2 = w.to_dict()

# Save extracted data dictionaries to JSON files
with open('liquidpr_ks_ecs_per_model.json', 'w') as f:
    json.dump(data2, f, indent=2)


## Timeseries and mixture PDF

In [ ]:
from portail_ing.risk.priors import load_sherwood_ecs
from matplotlib.ticker import ScalarFormatter

fig, axs = plt.subplot_mosaic(
    [['(a)'], ['(b)']], 
    figsize=(5.9, 5),
    layout='constrained',
)


percentiles = [10, 50, 90]

#---- plot timeseries ----

plot = {'Observations': inds['ahccd'][cind].sel(time=slice("1950", "2020"))}

exp_per = futs[cind].experiment_percentiles(percentiles)

# Historical period
plot['Historical'] = exp_per.sel(time=slice('1950', '2015'), experiment_id='ssp245')

# Simulations by SSP
groups = inds['espo'][cind].groupby('experiment_id')
for label, da in groups:
        plot[label] = exp_per.sel(time=slice('2015', '2100'), experiment_id=label)
    

# Use Figanos to plot the different time series
fg.plot.timeseries(
    plot,
    ax=axs['(a)'],
    show_lat_lon=False,
    plot_kw= {
        'Observations': {'linestyle': '--', 'color': 'k'},
        'Historical': {'color': '#18BBBB',},
    },
    legend=None,
)

#ax.set_title(title, loc="left")
axs['(a)'].set_title("a) Timeseries", loc='left')
axs['(a)'].set_ylabel("Total rainfall (mm)")
axs['(a)'].set_xlabel("Time")
axs['(a)'].legend(loc ='upper left', ncols=2, bbox_to_anchor=(0, 1.11))


# --- historical histogram + fit dist ----
axs['(b)'].hist(obs[cind].sample, density=True, alpha=.2, rwidth=.9, color=colors[0], 
                 label='Observed Histogram',
                 zorder=0,
                )

range_min = refs[cind].sample.min().item() - 0.05 * (refs[cind].sample.max().item() - refs[cind].sample.min().item())
range_max = refs[cind].sample.max().item() + 0.05 * (refs[cind].sample.max().item() - refs[cind].sample.min().item())
x = np.linspace(range_min, range_max, 500)
x = xr.DataArray(data=x, dims=(obs[cind].sample.name,), coords={obs[cind].sample.name: x})
axs['(b)'].plot(x, obs[cind].pdf(x), label='Observed', linewidth=2.5)

axs['(b)'].set_title('(b) Probability density functions', loc='left')
axs['(b)'].set_ylabel('Probability density (°C$^{-1}$)')

# --- Reference Mixture and pdfs ---
#x = np.linspace(refs[cind].sample.min().item(), refs[cind].sample.max().item(), 100)
vals = refs[cind].pdf(x).values

#Indiviually compute PDF for visualization
from portail_ing.risk.mixture import parametric_pdf
res = parametric_pdf(refs[cind].dparams.stack(realization=('source_id', 'experiment_id')), x)

lab1 = None
lab2 = None

for rea in res.realization.values:
    plt_kw = {
        "alpha": 0.3,
        "linewidth": 0.5,
        "zorder": 1,
    }
    if w.loc[rea[0]]['KS test ($w_1$)'] == 0:
        col =  'darkorange'
        if not lab1:
            lab1 = "KS rejected models (ref)"
            plt_kw['label'] = lab1
    else:
        col = 'grey'
        if not lab2:
            lab2 = "Models (ref)"
            plt_kw['label'] = lab2    
    plt_kw['color'] = col  

    axs['(b)'].plot(x, res.sel(realization=rea).values, **plt_kw)

axs['(b)'].plot(x, vals, label='Mixture (ref)', linewidth=2.5)


# --- Futre mixture PDF ---

xf = np.linspace(futs[cind].sample.min().item(), futs[cind].sample.max().item(), 100)
valsf = futs[cind].pdf(xf).values
axs['(b)'].plot(xf, valsf, label='Mixture (fut)', linewidth=2.5)
axs['(b)'].axvline(760, color='black', linestyle='--', label=f'Threshold = 760 mm', zorder=2)

axs['(b)'].set_xlabel('Total rainfall (mm)')
axs['(b)'].set_ylim(bottom=0)
axs['(b)'].set_xlim(400, 1750)

axs['(b)'].legend(loc ='upper right', bbox_to_anchor=(1.05, 1.2))


axs['(b)'].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
axs['(b)'].tick_params(axis='x', pad=5)

# have the right colors in eps format
axs['(a)'].set_rasterized(True)
axs['(b)'].set_rasterized(True)

plt.savefig('liquidpr_timeseries.pdf', format='pdf', dpi=600)   

In [ ]:
# figure data 
data = {}
for line in axs['(a)'].get_lines():
    data[line.get_label()] = {
        'x': line.get_xdata().astype(str).tolist(),
        'y': line.get_ydata().tolist(),
    }

data2 = {}
for line in axs['(b)'].get_lines():
    if line.get_label() in ['Observed', 'Mixture (ref)', 'Mixture (fut)']:
        data2[line.get_label()] = {
           'x': line.get_xdata().tolist(),
           'y': line.get_ydata().tolist(),
       }

# Save extracted data dictionaries to JSON files
with open('liquidpr_timeseries.json', 'w') as f:
    json.dump(data, f, indent=2)

with open('liquidpr_mixtures_pdf.json', 'w') as f:
    json.dump(data2, f, indent=2)

In [ ]:
import portail_ing.frontend.parameters
from importlib import reload

reload(portail_ing.frontend.parameters)
from portail_ing.frontend.parameters import weight_figure

fig1 = weight_figure(refs['cooling_degree_days'], futs['cooling_degree_days'])

In [ ]:
import portail_ing.frontend.parameters
from importlib import reload

reload(portail_ing.frontend.parameters)
from portail_ing.frontend.parameters import timeserie_mixture_figure

fig2 = timeserie_mixture_figure(
    obs['cooling_degree_days'],
    refs['cooling_degree_days'],
    futs['cooling_degree_days'],
    threshold=500,
    xlim=(75, 1100)
)